# Serving LLMs Efficiently with vLLM - Part II

In this notebook, you'll learn how to:
1. Launch a **vLLM inference server** serving a real model
2. Query it using the OpenAI-compatible API
3. **Explore model behavior** with logprobs and sampling parameters
4. Observe **continuous batching** and **KV cache** usage via live Prometheus metrics
5. Compare **prefix caching** on and off by output tokens/s
6. Compare **vLLM vs Ollama** throughput as concurrency increases


## From Quantization to Serving

In an earlier lesson, you used `llm-compressor` to quantize a model's weights from 16-bit to 4-bit using GPTQ. Now you'll **serve** a model with vLLM and interact with it through its OpenAI-compatible API.

[vLLM](https://github.com/vllm-project/vllm) is an open-source LLM inference engine that integrates key serving optimizations:

| Feature | Benefit |
|:--|:--|
| **Continuous batching** | Schedules at the token level: no wasted compute waiting for the longest request |
| **PagedAttention** | Manages KV cache in fixed-size blocks: near-zero memory waste |
| **Prefix caching** | Reuses KV cache for shared prompt prefixes across requests |
| **Quantization support** | Natively serves GPTQ, AWQ, and compressed-tensors models |
| **OpenAI-compatible API** | Drop-in replacement for applications already using the OpenAI client |

## Step 1: Start Your vLLM Server

In this learning environment, a pre-warmed vLLM server is provided to you to use with Qwen3-0.6B, so the server is already running and you'll use the code cells in this notebook to interact with it.

**How to serve a model with vLLM?**

To serve your model with vLLM, you'd need to run this command in the terminal. 

```bash
vllm serve Qwen/Qwen3-0.6B --dtype=bfloat16 --max-model-len 4096
```

You can find the installation details [here](https://docs.vllm.ai/en/latest/getting_started/installation/).

- **`vllm serve`**: launches vLLM's built-in inference server. It loads the model weights into the engine (with PagedAttention, continuous batching, and prefix caching enabled by default) and exposes it over HTTP on port 8000.
- **`Qwen/Qwen3-0.6B`**: the model identifier on the [Hugging Face Hub](https://huggingface.co/Qwen/Qwen3-0.6B). On first run, vLLM downloads the weights, tokenizer, and config from Hugging Face into the local cache (`~/.cache/huggingface/hub`), then loads them into memory. Subsequent runs reuse the cached files.
- **`--dtype=bfloat16`**: loads the weights in BF16 precision. 
- **`--max-model-len 4096`**: caps the context window (prompt + generation) at 4096 tokens. 

> **Note:** You can also serve the quantized model from the previous lesson, but that requires a GPU because its W4A16 format only has optimized runtime support on GPUs. Since this learning environment runs on CPU, the original model is the one served through vLLM.

vLLM wraps the model in an **OpenAI-compatible HTTP API**. It implements the same routes the OpenAI SDK calls (`/v1/models`, `/v1/chat/completions`, `/v1/completions`, `/v1/embeddings`) with the same request and response shapes.

So here let's check if it's running. 

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time, requests, json, os, math, sys

VLLM_URL = "http://localhost:8000"
os.makedirs("outputs", exist_ok=True)

print("Waiting for vLLM server...")
for attempt in range(60):
    try:
        r = requests.get(f"{VLLM_URL}/v1/models", timeout=5)
        if r.status_code == 200:
            MODEL = r.json()["data"][0]["id"]
            break
    except requests.ConnectionError:
        pass
    time.sleep(5)
    if attempt % 6 == 5:
        print(f"  Still waiting... ({(attempt + 1) * 5}s elapsed)")
else:
    raise RuntimeError(
        "vLLM server not reachable after 5 minutes."
    )

print(f"Connected to {VLLM_URL} — model: {MODEL}")

> **Note:** The vLLM server might need 1 or 2 minutes to be ready.

## Your First Local LLM Request

vLLM exposes an OpenAI-compatible API, so we use the standard `openai` Python client.

In [ ]:
from openai import OpenAI
client = OpenAI(base_url=f"{VLLM_URL}/v1", api_key="unused")

Qwen3 supports a *thinking mode* that generates chain-of-thought reasoning before answering. We disable it with `enable_thinking: False` to keep responses short and predictable.

In [ ]:
start = time.time()
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", 
               "content": "What is PagedAttention in one sentence?"}],
    max_tokens=80,
    temperature=0.7,
    top_p=0.8,
    extra_body={"top_k": 20, 
                "chat_template_kwargs": {"enable_thinking": False}},
)
elapsed = time.time() - start

print(f"Response ({elapsed:.2f}s, {resp.usage.completion_tokens} tokens):")
print(resp.choices[0].message.content)
print(f"\nUsage: {resp.usage.prompt_tokens} prompt + "
      f"{resp.usage.completion_tokens} completion = {resp.usage.total_tokens} total")

## Exploring Model Behavior

Beyond simple chat, the vLLM API lets you inspect model internals and control generation. For example, **Logprobs:** allows you to see the model's confidence in each token it generates, plus the alternatives it considered

In [ ]:
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "The capital of France is"}],
    max_tokens=15,
    temperature=0.0,
    logprobs=True,
    top_logprobs=5,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

print(f"Response: {resp.choices[0].message.content.strip()}\n")
print("Token-by-token probabilities:\n")

for tok in resp.choices[0].logprobs.content[:8]:
    print(f"  Chosen: '{tok.token}'  (logprob {tok.logprob:.2f})")
    if tok.top_logprobs:
        for alt in tok.top_logprobs[:5]:
            pct = math.exp(alt.logprob) * 100
            bar = "\u2588" * min(20, max(1, int(pct / 5)))
            print(f"    {pct:5.1f}%  {bar}  '{alt.token}'")
    print()

## Observing vLLM Under the Hood

vLLM exposes a Prometheus-compatible **`/metrics`** endpoint (it's a format that is easy to scrape). Key metrics to watch:

- **`num_requests_running / waiting`**: how many requests are active vs queued
- **`gpu_cache_usage_perc`** (or `cpu_cache_usage_perc`): KV cache memory pressure
- **`prompt_tokens_total / generation_tokens_total`**: cumulative token counts


In [ ]:
def get_vllm_metrics(base_url=VLLM_URL):
    """Scrape vLLM Prometheus /metrics and return {name: value}."""
    r = requests.get(f"{base_url}/metrics")
    metrics = {}
    for line in r.text.split("\n"):
        if line.startswith("#") or not line.strip():
            continue
        name = line.split("{")[0].split()[0]
        try:
            metrics[name] = float(line.split()[-1])
        except (ValueError, IndexError):
            continue
    return metrics

metrics = get_vllm_metrics()
print("Current vLLM Metrics:")
for key in ["vllm:num_requests_running", "vllm:num_requests_waiting",
            "vllm:gpu_cache_usage_perc", "vllm:cpu_cache_usage_perc",
            "vllm:prompt_tokens_total", "vllm:generation_tokens_total"]:
    if key in metrics:
        print(f"  {key.replace('vllm:', '')}: {metrics[key]:g}")

with open("outputs/metrics_snapshot.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\nFull metrics saved to outputs/metrics_snapshot.json")

## Continuous Batching in Action

vLLM uses **continuous batching** (iteration-level scheduling): when a request finishes generating, its slot is immediately filled by the next waiting request.

Let's send 5 requests **concurrently** and watch the metrics while they run.

In [ ]:
import concurrent.futures

prompts = [
    "What is quantization?",
    "Explain KV caching briefly.",
    "What is continuous batching?",
    "Why is LLM inference memory-bound?",
    "What is PagedAttention?",
]

def _ask(prompt):
    return client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=60, temperature=0.7,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )

before = get_vllm_metrics()
print(f"Sending {len(prompts)} concurrent requests...\n")
start = time.time()

with concurrent.futures.ThreadPoolExecutor(
    max_workers=len(prompts)) as pool:
    futures = {pool.submit(_ask, p): p for p in prompts}
    time.sleep(0.5)
    during = get_vllm_metrics()
    running = during.get("vllm:num_requests_running", "--")
    waiting = during.get("vllm:num_requests_waiting", "--")
    print(f"  [mid-flight]  running: {running}  |  waiting: {waiting}")

    for f in concurrent.futures.as_completed(futures):
        resp = f.result()
        print(f"  done: \"{futures[f][:40]}\" -> {resp.usage.completion_tokens} tokens")

elapsed = time.time() - start
after = get_vllm_metrics()
tokens = after.get("vllm:generation_tokens_total", 0) - before.get(
    "vllm:generation_tokens_total", 0)

print(f"\nAll {len(prompts)} completed in {elapsed:.2f}s")
if tokens > 0:
    print(f"Tokens generated: {tokens:g}  |  ~{tokens / elapsed:.1f} tokens/s")

>**Note**: The first time you run this cell, you may see running: 0 or running: 1 instead of 5. Run the cell again and it will show 5. The metric is checked just once, a fraction of a second after sending the requests. That single reading is taken so soon that the requests may not have reached the server yet, so it can show 0 (nothing running and nothing queued) even though all five are about to run together. Sampling repeatedly instead shows the true peak of 5.

**What Just Happened**

vLLM's scheduler received all 5 requests and processed them using **continuous batching**. Each generation step produces tokens for every request in the batch, and completed requests free their slots immediately.

vLLM's **PagedAttention** is key to making this work at scale: the KV cache is divided into fixed-size blocks that can be placed anywhere in memory. When a request finishes, its blocks are immediately available for reuse — no fragmentation.

## Prefix Caching

Many applications send the same long **system prompt** with every request. **Prefix caching** is on by default: the first request builds the KV cache for that shared prefix, and later requests reuse it. Only the new question and the short answer are new work, so those later requests finish sooner and **output tokens/s** goes up.

This machine can run only one vLLM server at a time, so the two settings are measured one after the other on port 8000. The benchmark uses a long repeated system prompt and caps each answer at a few tokens, so prefill is most of the request. That is the work prefix caching skips after the first call, which shows up as a lower time to first token and higher output tokens/s.

**Step 1.** Start the server with prefix caching off, wait until it is ready, then run the next cell:

```bash
vllm serve Qwen/Qwen3-0.6B --dtype=bfloat16 --max-model-len 4096 --no-enable-prefix-caching
```


In [ ]:
import json
import os
import time

import requests
from openai import OpenAI

# Run this while vLLM was started with --no-enable-prefix-caching.
# It only talks to the server already on port 8000 and saves the timings.
if "VLLM_URL" not in globals():
    VLLM_URL = "http://localhost:8000"

SHARED_BLOCK = (
    "You are a helpful AI teaching assistant for a course on "
    "LLM optimization. Cover quantization, the KV cache, continuous "
    "batching, PagedAttention, and model serving. "
    "Answer in one short sentence.\n"
)
# Long shared prefix, short answer: prefill is most of the request.
# Lower PREFIX_REPEATS if a request exceeds the server context length.
PREFIX_REPEATS = 60
SYSTEM_PROMPT = SHARED_BLOCK * PREFIX_REPEATS
MAX_TOKENS = 8
OFF = "Prefix caching off"
OFF_PATH = "outputs/prefix_caching_off.json"
EXTRA = {"chat_template_kwargs": {"enable_thinking": False}}

questions = [
    "What is weight quantization?",
    "How does vLLM handle memory?",
    "What is continuous batching?",
    "Why use prefix caching?",
    "What is GPTQ?",
]


def active_model():
    try:
        response = requests.get(f"{VLLM_URL}/v1/models", timeout=5)
        response.raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError(
            f"No vLLM server at {VLLM_URL}. "
            "Start it with --no-enable-prefix-caching, then rerun this cell."
        ) from exc
    return response.json()["data"][0]["id"]


def read_counters(base_url):
    """Sum prefix-cache hit and query counters. Ignore Prometheus *_created gauges."""
    try:
        text = requests.get(f"{base_url}/metrics", timeout=10).text
    except requests.RequestException:
        return None
    totals = {"hits": 0.0, "queries": 0.0}
    found = False
    for line in text.splitlines():
        if not line or line.startswith("#"):
            continue
        key = line.split("{", 1)[0].split()[0]
        if key.endswith("prefix_cache_hits") or key.endswith("prefix_cache_hits_total"):
            bucket = "hits"
        elif key.endswith("prefix_cache_queries") or key.endswith("prefix_cache_queries_total"):
            bucket = "queries"
        else:
            continue
        try:
            totals[bucket] += float(line.split()[-1])
            found = True
        except ValueError:
            continue
    return totals if found else None


def measure_one(chat_client, model_name, mode, request_idx, question):
    """Stream so time-to-first-token is the prefill, separate from the short answer."""
    started = time.perf_counter()
    ttft = None
    usage = None
    stream = chat_client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
        max_tokens=MAX_TOKENS,
        temperature=0,
        top_p=1,
        stream=True,
        stream_options={"include_usage": True},
        extra_body=EXTRA,
    )
    for chunk in stream:
        choice = chunk.choices[0] if chunk.choices else None
        if (
            ttft is None
            and choice is not None
            and choice.delta is not None
            and choice.delta.content
        ):
            ttft = time.perf_counter() - started
        if getattr(chunk, "usage", None) is not None:
            usage = chunk.usage
    elapsed = time.perf_counter() - started
    if ttft is None:
        ttft = elapsed
    if usage is None or usage.completion_tokens is None:
        raise RuntimeError(f"{mode} did not return a completion token count.")
    completion = usage.completion_tokens
    return {
        "mode": mode,
        "request": request_idx,
        "question": question,
        "prompt_tokens": usage.prompt_tokens,
        "completion_tokens": completion,
        "seconds": elapsed,
        "ttft_s": ttft,
        "tokens_per_s": completion / elapsed if elapsed else 0.0,
    }


def warmup(chat_client, model_name):
    chat_client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": "Reply with the single word: ready"}],
        max_tokens=8,
        temperature=0,
        extra_body=EXTRA,
    )


def run_mode(mode, chat_client, model_name, base_url):
    print(mode, flush=True)
    print("  warmup...", flush=True)
    warmup(chat_client, model_name)
    before = read_counters(base_url)
    rows = []
    for i, question in enumerate(questions, start=1):
        row = measure_one(chat_client, model_name, mode, i, question)
        rows.append(row)
        prompt = f"  prompt {row['prompt_tokens']} tok" if i == 1 else ""
        print(
            f"  [{i}] {question:<32} ttft {row['ttft_s']:.2f}s  "
            f"{row['seconds']:.2f}s total  "
            f"{row['completion_tokens']} tok  {row['tokens_per_s']:.1f} tok/s{prompt}",
            flush=True,
        )
    after = read_counters(base_url)
    hits = None
    if before and after:
        hits = after["hits"] - before["hits"]
        queries = after["queries"] - before["queries"]
        print(f"  prefix-cache queries +{queries:.0f}, hits +{hits:.0f}")
    else:
        print("  prefix-cache counters were not on /metrics")
    return rows, hits


model_name = active_model()
bench_client = OpenAI(
    base_url=f"{VLLM_URL}/v1", api_key="unused", timeout=300, max_retries=0,
)
print(f"Server {VLLM_URL}  model {model_name}")
print("Expect this process to be the one started with --no-enable-prefix-caching.")

off_rows, off_hits = run_mode(OFF, bench_client, model_name, VLLM_URL)
os.makedirs("outputs", exist_ok=True)
with open(OFF_PATH, "w") as f:
    json.dump(
        {
            "mode": OFF,
            "system_prompt": SYSTEM_PROMPT,
            "questions": questions,
            "max_tokens": MAX_TOKENS,
            "hits": off_hits,
            "rows": off_rows,
        },
        f,
        indent=2,
    )

steady_tps = [row["tokens_per_s"] for row in off_rows if row["request"] > 1]
steady_ttft = [row["ttft_s"] for row in off_rows if row["request"] > 1]
print()
print(f"Requests 2-5 time to first token: {sum(steady_ttft) / len(steady_ttft):.2f}s")
print(f"Requests 2-5 output tokens/s: {sum(steady_tps) / len(steady_tps):.1f}")
if off_hits is not None and off_hits > 0:
    print(
        "This server recorded prefix-cache hits. "
        "Restart it with --no-enable-prefix-caching and run this cell again."
    )
print(f"Saved {OFF_PATH}")
print()
print("Stop this server. Start it again with prefix caching on (the default serve command).")
print("When port 8000 is ready, run the next code cell.")


**Step 2.** Stop the server from step 1. Start it again with prefix caching on (leave out `--no-enable-prefix-caching`):

```bash
vllm serve Qwen/Qwen3-0.6B --dtype=bfloat16 --max-model-len 4096
```

Wait until port 8000 is ready, then run the next cell. It repeats the same prompt and plots both runs. The kernel can stay up while you restart vLLM; the next cell also works after a kernel restart because it reads `outputs/prefix_caching_off.json`.


In [ ]:
import json
import os
import time

import matplotlib.pyplot as plt
import requests
from openai import OpenAI

# Run this after stopping the no-cache server and starting vLLM with prefix caching on.
if "VLLM_URL" not in globals():
    VLLM_URL = "http://localhost:8000"

ON = "Prefix caching on"
OFF = "Prefix caching off"
OFF_PATH = "outputs/prefix_caching_off.json"
EXTRA = {"chat_template_kwargs": {"enable_thinking": False}}

if not os.path.exists(OFF_PATH):
    raise RuntimeError(
        "Missing outputs/prefix_caching_off.json. "
        "Run the previous code cell while the server is started with "
        "--no-enable-prefix-caching."
    )
with open(OFF_PATH) as f:
    saved = json.load(f)

SYSTEM_PROMPT = saved["system_prompt"]
questions = saved["questions"]
MAX_TOKENS = saved["max_tokens"]
off_rows = saved["rows"]
off_hits = saved.get("hits")


def active_model():
    try:
        response = requests.get(f"{VLLM_URL}/v1/models", timeout=5)
        response.raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError(
            f"No vLLM server at {VLLM_URL}. "
            "Start it with prefix caching on, then rerun this cell."
        ) from exc
    return response.json()["data"][0]["id"]


def read_counters(base_url):
    try:
        text = requests.get(f"{base_url}/metrics", timeout=10).text
    except requests.RequestException:
        return None
    totals = {"hits": 0.0, "queries": 0.0}
    found = False
    for line in text.splitlines():
        if not line or line.startswith("#"):
            continue
        key = line.split("{", 1)[0].split()[0]
        if key.endswith("prefix_cache_hits") or key.endswith("prefix_cache_hits_total"):
            bucket = "hits"
        elif key.endswith("prefix_cache_queries") or key.endswith("prefix_cache_queries_total"):
            bucket = "queries"
        else:
            continue
        try:
            totals[bucket] += float(line.split()[-1])
            found = True
        except ValueError:
            continue
    return totals if found else None


def measure_one(chat_client, model_name, mode, request_idx, question):
    """Stream so time-to-first-token is the prefill, separate from the short answer."""
    started = time.perf_counter()
    ttft = None
    usage = None
    stream = chat_client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
        max_tokens=MAX_TOKENS,
        temperature=0,
        top_p=1,
        stream=True,
        stream_options={"include_usage": True},
        extra_body=EXTRA,
    )
    for chunk in stream:
        choice = chunk.choices[0] if chunk.choices else None
        if (
            ttft is None
            and choice is not None
            and choice.delta is not None
            and choice.delta.content
        ):
            ttft = time.perf_counter() - started
        if getattr(chunk, "usage", None) is not None:
            usage = chunk.usage
    elapsed = time.perf_counter() - started
    if ttft is None:
        ttft = elapsed
    if usage is None or usage.completion_tokens is None:
        raise RuntimeError(f"{mode} did not return a completion token count.")
    completion = usage.completion_tokens
    return {
        "mode": mode,
        "request": request_idx,
        "question": question,
        "prompt_tokens": usage.prompt_tokens,
        "completion_tokens": completion,
        "seconds": elapsed,
        "ttft_s": ttft,
        "tokens_per_s": completion / elapsed if elapsed else 0.0,
    }


def warmup(chat_client, model_name):
    chat_client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": "Reply with the single word: ready"}],
        max_tokens=8,
        temperature=0,
        extra_body=EXTRA,
    )


def run_mode(mode, chat_client, model_name, base_url):
    print(mode, flush=True)
    print("  warmup...", flush=True)
    warmup(chat_client, model_name)
    before = read_counters(base_url)
    rows = []
    for i, question in enumerate(questions, start=1):
        row = measure_one(chat_client, model_name, mode, i, question)
        rows.append(row)
        prompt = f"  prompt {row['prompt_tokens']} tok" if i == 1 else ""
        print(
            f"  [{i}] {question:<32} ttft {row['ttft_s']:.2f}s  "
            f"{row['seconds']:.2f}s total  "
            f"{row['completion_tokens']} tok  {row['tokens_per_s']:.1f} tok/s{prompt}",
            flush=True,
        )
    after = read_counters(base_url)
    hits = None
    if before and after:
        hits = after["hits"] - before["hits"]
        queries = after["queries"] - before["queries"]
        print(f"  prefix-cache queries +{queries:.0f}, hits +{hits:.0f}")
    else:
        print("  prefix-cache counters were not on /metrics")
    return rows, hits


def steady(rows, key):
    picked = [row[key] for row in rows if row["request"] > 1]
    return sum(picked) / len(picked)


model_name = active_model()
bench_client = OpenAI(
    base_url=f"{VLLM_URL}/v1", api_key="unused", timeout=300, max_retries=0,
)
print(f"Server {VLLM_URL}  model {model_name}")
print("Expect this process to be the one started with prefix caching on.")

on_rows, on_hits = run_mode(ON, bench_client, model_name, VLLM_URL)
prefix_rows = off_rows + on_rows
os.makedirs("outputs", exist_ok=True)
with open("outputs/prefix_caching.json", "w") as f:
    json.dump(prefix_rows, f, indent=2)

if "ttft_s" not in off_rows[0]:
    raise RuntimeError(
        "The saved run has no time-to-first-token. Re-run the previous code cell "
        "while the server is started with --no-enable-prefix-caching."
    )

off_ttft = steady(off_rows, "ttft_s")
on_ttft = steady(on_rows, "ttft_s")
off_tps = steady(off_rows, "tokens_per_s")
on_tps = steady(on_rows, "tokens_per_s")
print()
if on_ttft > 0:
    print(
        f"Requests 2-5 time to first token: off {off_ttft:.2f}s, "
        f"on {on_ttft:.2f}s ({off_ttft / on_ttft:.1f}x sooner with caching)"
    )
else:
    print(f"Requests 2-5 time to first token: off {off_ttft:.2f}s, on {on_ttft:.2f}s")
if off_tps > 0:
    print(
        f"Requests 2-5 output tokens/s:     off {off_tps:.1f}, "
        f"on {on_tps:.1f} ({on_tps / off_tps:.1f}x with caching)"
    )
else:
    print(f"Requests 2-5 output tokens/s:     off {off_tps:.1f}, on {on_tps:.1f}")
if on_hits is not None and on_hits <= 0:
    print(
        "This server recorded no prefix-cache hits. "
        "It may still be the process started with --no-enable-prefix-caching."
    )
print("Saved outputs/prefix_caching.json")

COLORS = {ON: "#0B6E4F", OFF: "#1B4F9C"}
MARKERS = {ON: "o", OFF: "s"}

with plt.rc_context({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.alpha": 0.45,
    "font.size": 11,
    "axes.axisbelow": True,
}):
    fig, axes = plt.subplots(2, 1, figsize=(8.2, 7.4), constrained_layout=True)
    fig.suptitle("Prefix caching on vs off — same model, same prompt", fontsize=14)

    def draw(ax, key, ylabel, title):
        for mode in (OFF, ON):
            rows = [row for row in prefix_rows if row["mode"] == mode]
            ax.plot(
                [row["request"] for row in rows],
                [row[key] for row in rows],
                label=mode,
                color=COLORS[mode],
                marker=MARKERS[mode],
                linewidth=2.25,
                markersize=8,
            )
        ax.set_title(title, loc="left", fontsize=12)
        ax.set_xlabel("Request number")
        ax.set_ylabel(ylabel)
        ax.set_xticks([row["request"] for row in on_rows])
        ax.set_ylim(bottom=0)
        ax.legend(frameon=False)

    draw(
        axes[0],
        "ttft_s",
        "Seconds",
        "Figure 1. Time to first token (lower is faster)",
    )
    draw(
        axes[1],
        "tokens_per_s",
        "Output tokens per second",
        "Figure 2. Output tokens/s (higher is faster)",
    )

    fig.savefig("outputs/prefix_caching_tokens_per_s.png", dpi=140, bbox_inches="tight")
    plt.show()

print("Saved outputs/prefix_caching_tokens_per_s.png")


**Why Prefix Caching Matters**

Request 1 in each run pays the full prefill cost for the shared system prompt, so the time to first token matches. From request 2 onward, prefix caching reuses that prefix: the first token arrives much sooner, and the same short answer finishes at a higher output tokens/s. The run started with `--no-enable-prefix-caching` rebuilds the prefix on every call, so both curves stay near the first request.

Figure 1 is the direct view of that savings. Figure 2 is the same runs as output tokens/s. Answers are capped at a few tokens so generation does not hide the prefill gap.


---

## (Optional) KV Cache Size Visualization for Qwen3-0.6B

The KV cache stores key and value tensors from the attention mechanism. It grows **linearly** with sequence length and concurrent requests.

In [ ]:
num_layers = 28
num_kv_heads = 8     # GQA: 16 Q heads, 8 KV heads
head_dim = 128
dtype_bytes = 2      # BF16

per_token = 2 * num_layers * num_kv_heads * head_dim * dtype_bytes

print(f"KV Cache -- Qwen3-0.6B")
print(f"Per token: 2 x {num_layers} x {num_kv_heads} x {head_dim} x {dtype_bytes}"
      f" = {per_token:,} bytes ({per_token // 1024} KB)\n")
print(f"  {'Context':>8}  {'KV Cache':>10}")
print(f"  {'-'*8}  {'-'*10}")
for ctx in [1, 64, 256, 1024, 4096]:
    size = per_token * ctx
    label = f"{size / 1024:.0f} KB" if size < 1024**2 else f"{size / 1024**2:.0f} MB"
    print(f"  {ctx:>7}t  {label:>10}")

print(f"\n  10 concurrent x 4096 ctx = {per_token * 4096 * 10 / 1024**3:.1f} GB")

---

## (Optional) Thinking Mode

Qwen3 supports a *thinking mode* where the model generates internal chain-of-thought reasoning (`<think>...</think>`) before the visible answer. This produces better answers but uses **significantly more tokens** — more KV cache, more compute, longer response.

Let's stream both modes side by side on the same prompt.

In [ ]:
prompt = "What makes continuous batching better than static batching?"

for label, thinking, max_tok in [
    ("Thinking OFF", False, 80), ("Thinking ON", True, 200)]:
    print(f"=== {label} ===\n")
    start = time.time()
    tokens = 0
    stream = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tok, temperature=0.7, stream=True,
        extra_body={"chat_template_kwargs": {"enable_thinking": thinking}},
    )
    for chunk in stream:
        if chunk.choices[0].delta.content:
            sys.stdout.write(chunk.choices[0].delta.content)
            sys.stdout.flush()
            tokens += 1
    elapsed = time.time() - start
    print(f"\n  [{tokens} tokens, {elapsed:.2f}s]\n")

## vLLM vs Ollama under Concurrency

vLLM and Ollama can serve the **same model** through an OpenAI-compatible API, but they schedule concurrent requests differently.

- **Ollama** is built for local, single-user use. Out of the box it only runs a few requests at once, so extra callers mostly wait in a queue.
- **vLLM** continuously batches whatever is in flight, so throughput can climb as more requests arrive together.

The charts follow the same idea as the [Red Hat Ollama vs vLLM benchmark](https://developers.redhat.com/articles/2025/08/08/ollama-vs-vllm-deep-dive-performance-benchmarking): one fixed prompt, a rising number of concurrent requests, then **output tokens per second** and **requests per second**. Each level sends only a few requests, so the sweep finishes in a few minutes.

Both servers are already running. The cells below call vLLM at `localhost:8000` and Ollama at `localhost:11434`, one engine at a time.


In [ ]:
import requests
from openai import OpenAI

OLLAMA_URL = "http://localhost:11434"
# Set this to an Ollama tag to skip auto-matching, for example "qwen3:0.6b".
OLLAMA_MODEL = None


def list_model_ids(base_url):
    """Return model ids from an OpenAI-compatible /v1/models endpoint."""
    try:
        response = requests.get(f"{base_url}/v1/models", timeout=10)
        response.raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError(
            f"Could not reach {base_url}/v1/models. "
            "Start that server, then rerun this cell."
        ) from exc
    return [item["id"] for item in response.json().get("data", [])]


def list_ollama_models(base_url):
    """Prefer Ollama's OpenAI endpoint, and fall back to /api/tags for names."""
    try:
        response = requests.get(f"{base_url}/v1/models", timeout=10)
        if response.status_code == 200:
            ids = [item["id"] for item in response.json().get("data", [])]
            if ids:
                return ids
    except requests.RequestException:
        pass
    try:
        response = requests.get(f"{base_url}/api/tags", timeout=10)
        response.raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError(
            f"Could not reach Ollama at {base_url}. "
            "Expected /v1/models (port 11434 by default)."
        ) from exc
    return [item["name"] for item in response.json().get("models", [])]


def _norm(name):
    return "".join(ch for ch in name.lower() if ch.isalnum())


def choose_ollama_model(vllm_model, ollama_models, override):
    if override:
        if override not in ollama_models:
            raise RuntimeError(
                f"OLLAMA_MODEL={override!r} is not loaded. Ollama has: {ollama_models}"
            )
        return override

    target = _norm(vllm_model.split("/")[-1])
    exact = [name for name in ollama_models if _norm(name) == target]
    if exact:
        return exact[0]

    partial = [
        name for name in ollama_models
        if target in _norm(name) or _norm(name) in target
    ]
    if partial:
        return min(partial, key=lambda name: abs(len(_norm(name)) - len(target)))

    if len(ollama_models) == 1:
        print(f"No name match for {vllm_model}. Using the only Ollama model: {ollama_models[0]}")
        return ollama_models[0]

    raise RuntimeError(
        "Could not match the vLLM model to an Ollama tag. "
        f"vLLM is serving {vllm_model}. Ollama has {ollama_models}. "
        "Set OLLAMA_MODEL to one of those tags and rerun this cell."
    )


vllm_models = list_model_ids(VLLM_URL)
ollama_models = list_ollama_models(OLLAMA_URL)
print("vLLM models:  ", vllm_models)
print("Ollama models:", ollama_models)

VLLM_MODEL = MODEL
OLLAMA_MODEL = choose_ollama_model(VLLM_MODEL, ollama_models, OLLAMA_MODEL)
if _norm(OLLAMA_MODEL) != _norm(VLLM_MODEL.split("/")[-1]):
    print(
        "These names are not an exact match. "
        "Set OLLAMA_MODEL above if this is not the same model."
    )

vllm_client = OpenAI(
    base_url=f"{VLLM_URL}/v1", api_key="unused", timeout=180, max_retries=0,
)
ollama_client = OpenAI(
    base_url=f"{OLLAMA_URL}/v1", api_key="ollama", timeout=180, max_retries=0,
)
print()
print(f"Comparing vLLM '{VLLM_MODEL}' with Ollama '{OLLAMA_MODEL}'")


In [ ]:
import concurrent.futures

from openai import APIStatusError

# Same prompt, same token cap, same greedy decoding on both engines.
# One engine at a time. Expect a few minutes.
BENCH_PROMPT = (
    "Count from 1 to 30, one number per line. Do not add any other words."
)
MAX_TOKENS = 64
CONCURRENCY_LEVELS = [1, 2, 4, 8]
REQUESTS_PER_USER = 2  # each in-flight slot sends this many requests back to back


def call_chat(client, model, extra_body):
    kwargs = {
        "model": model,
        "messages": [{"role": "user", "content": BENCH_PROMPT}],
        "max_tokens": MAX_TOKENS,
        "temperature": 0,
        "top_p": 1,
    }
    if extra_body:
        kwargs["extra_body"] = extra_body
    started = time.perf_counter()
    response = client.chat.completions.create(**kwargs)
    elapsed = time.perf_counter() - started
    usage = response.usage
    if usage is None or usage.completion_tokens is None:
        raise RuntimeError(f"{model} did not return a completion token count.")
    return elapsed, usage.completion_tokens


def benchmark_engine(engine, client, model, extra_body):
    """Keep `concurrency` requests in flight and measure completed work / wall time."""
    print()
    print(f"{engine}  model={model}")
    try:
        warmup_s, warmup_tokens = call_chat(client, model, extra_body)
    except APIStatusError as exc:
        if extra_body and exc.status_code in (400, 422):
            print("  server rejected extra options; retrying without them")
            extra_body = None
            warmup_s, warmup_tokens = call_chat(client, model, extra_body)
        else:
            raise
    print(f"  warmup: {warmup_s:.2f}s, {warmup_tokens} completion tokens")

    rows = []
    for concurrency in CONCURRENCY_LEVELS:
        def one_user(_slot, body=extra_body):
            latencies = []
            tokens = 0
            for _ in range(REQUESTS_PER_USER):
                elapsed, completion_tokens = call_chat(client, model, body)
                latencies.append(elapsed)
                tokens += completion_tokens
            return latencies, tokens

        n_requests = concurrency * REQUESTS_PER_USER
        print(f"  concurrency {concurrency}: {n_requests} requests ...", flush=True)
        started = time.perf_counter()
        with concurrent.futures.ThreadPoolExecutor(max_workers=concurrency) as pool:
            parts = list(pool.map(one_user, range(concurrency)))
        wall_s = time.perf_counter() - started

        latencies = [dt for part, _ in parts for dt in part]
        output_tokens = sum(tokens for _, tokens in parts)
        row = {
            "engine": engine,
            "model": model,
            "concurrency": concurrency,
            "requests": n_requests,
            "wall_s": round(wall_s, 3),
            "output_tokens": output_tokens,
            "tps": output_tokens / wall_s,
            "rps": n_requests / wall_s,
            "avg_latency_s": sum(latencies) / len(latencies),
        }
        rows.append(row)
        print(f"    {row['tps']:.1f} tok/s   {row['rps']:.2f} req/s   wall {wall_s:.1f}s")
    return rows, warmup_tokens


engines = [
    (
        "vLLM",
        vllm_client,
        VLLM_MODEL,
        {"chat_template_kwargs": {"enable_thinking": False}},
    ),
    (
        "Ollama",
        ollama_client,
        OLLAMA_MODEL,
        {"think": False},
    ),
]

os.makedirs("outputs", exist_ok=True)
bench_results = []
warmup_tokens = {}
for engine, engine_client, model, extra in engines:
    rows, warmup = benchmark_engine(engine, engine_client, model, extra)
    warmup_tokens[engine] = warmup
    bench_results.extend(rows)
    with open("outputs/vllm_vs_ollama.json", "w") as f:
        json.dump(bench_results, f, indent=2)

v_tok, o_tok = warmup_tokens["vLLM"], warmup_tokens["Ollama"]
print()
print(f"Warmup completion tokens — vLLM: {v_tok}, Ollama: {o_tok}")
if min(v_tok, o_tok) > 0 and max(v_tok, o_tok) > 2 * min(v_tok, o_tok):
    print(
        "Those lengths differ by more than 2x, so one engine is generating "
        "much more text per request (often thinking mode). The chart still "
        "shows measured throughput, but the per-request work is not equal."
    )
print("Saved outputs/vllm_vs_ollama.json")


In [ ]:
import matplotlib.pyplot as plt

ENGINES = ["vLLM", "Ollama"]
COLORS = {"vLLM": "#CC0000", "Ollama": "#1B4F9C"}
MARKERS = {"vLLM": "o", "Ollama": "s"}


def rows_for(engine):
    return sorted(
        (row for row in bench_results if row["engine"] == engine),
        key=lambda row: row["concurrency"],
    )


print(
    f"{'Engine':<8} {'Concurrency':>12} {'Tokens/s':>10} "
    f"{'Requests/s':>12} {'Avg latency (s)':>16}"
)
print("-" * 62)
for engine in ENGINES:
    for row in rows_for(engine):
        print(
            f"{row['engine']:<8} {row['concurrency']:>12} "
            f"{row['tps']:>10.1f} {row['rps']:>12.2f} {row['avg_latency_s']:>16.2f}"
        )

v_tps = rows_for("vLLM")[-1]["tps"]
o_tps = rows_for("Ollama")[-1]["tps"]
hi = rows_for("vLLM")[-1]["concurrency"]
if o_tps > 0:
    print(
        f"At {hi} concurrent requests: "
        f"vLLM {v_tps:.1f} tok/s, Ollama {o_tps:.1f} tok/s "
        f"({v_tps / o_tps:.2f}x)."
    )

with plt.rc_context({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.alpha": 0.45,
    "font.size": 11,
    "axes.axisbelow": True,
}):
    fig, axes = plt.subplots(2, 1, figsize=(8.2, 7.4), sharex=True, constrained_layout=True)
    fig.suptitle("vLLM vs Ollama — same model, rising concurrency", fontsize=14)
    panels = [
        ("tps", "Output tokens per second", "Figure 1. Output token throughput"),
        ("rps", "Requests per second", "Figure 2. Request throughput"),
    ]
    levels = [row["concurrency"] for row in rows_for("vLLM")]
    for ax, (key, ylabel, title) in zip(axes, panels):
        for engine in ENGINES:
            rows = rows_for(engine)
            ax.plot(
                [row["concurrency"] for row in rows],
                [row[key] for row in rows],
                label=engine,
                color=COLORS[engine],
                marker=MARKERS[engine],
                linewidth=2.25,
                markersize=8,
            )
        ax.set_ylabel(ylabel)
        ax.set_title(title, loc="left", fontsize=12)
        ax.set_xticks(levels)
        ax.set_ylim(bottom=0)
        ax.legend(frameon=False)
    axes[-1].set_xlabel("Concurrent requests")
    os.makedirs("outputs", exist_ok=True)
    fig.savefig(
        "outputs/vllm_vs_ollama_throughput.png", dpi=140, bbox_inches="tight",
    )
    plt.show()

print("Saved outputs/vllm_vs_ollama_throughput.png")


**How to read the charts**

Both charts use the same concurrency levels. **Tokens per second** is total output tokens divided by wall-clock time while that many requests are kept in flight. **Requests per second** is the same run counted in completed calls.

A line that climbs is an engine turning extra concurrency into extra throughput. A line that flattens has hit its parallelism limit: more callers wait, but the model is not doing more work at once. That is the usual gap between vLLM's continuous batching and Ollama's local-serving defaults.


## Summary

In this notebook, you:

- Launched a **vLLM server** serving Qwen3-0.6B and connected with the OpenAI client
- Explored model internals with **logprobs** (token probabilities)
- Used the **`/metrics` endpoint** to observe KV cache usage and request counts
- Demonstrated **continuous batching**: sending concurrent requests
- Compared **prefix caching** on and off: the same long prompt produces more output tokens/s when caching stays on
- Streamed **thinking mode** to see chain-of-thought reasoning live
- Compared **vLLM and Ollama** on the same model as concurrent requests increase


## Resources

- [vLLM Github](https://github.com/vllm-project/vllm)
- [vLLM Docs](https://docs.vllm.ai/en/latest/)
- [vLLM Installation Instructions](https://docs.vllm.ai/en/latest/getting_started/installation/)
- [Ollama](https://ollama.com/)
- [Ollama vs vLLM benchmark (Red Hat Developer)](https://developers.redhat.com/articles/2025/08/08/ollama-vs-vllm-deep-dive-performance-benchmarking)
